# Prepare environment

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

# Load data

In [ ]:
from src.dataset.digit import DigitDataModule

data_dir = "../../data/digit/"
batch_size = 32
num_workers = 0
dm = DigitDataModule(data_dir, batch_size=batch_size, num_workers=num_workers)

# Visualize learning rate

In [ ]:
%matplotlib inline

import lightning as L
from lightning.pytorch.tuner import Tuner
import matplotlib.pyplot as plt
import torch

from model.torch_digit_classifier import DigitClassifier
from model.digit_classifier import Classifier

# Model parameters
num_classes = 10
feature_size = [28, 28]
feature_map_sizes = [6, 20]
conv_kernel_sizes = [5, 5]
conv_dropout_probs = [0.1, 0.1]
pool_kernel_sizes = [2, 2]
pool_strides = [2, 2]
fc_sizes = [100, 10]
fc_dropout_probs = [0.2, 0.2]

# Learning parameters
learning_rate = 0.001
weight_decay = 0.01

torch.manual_seed(42)

pytorch_model = DigitClassifier(
    feature_size,
    feature_map_sizes,
    conv_kernel_sizes,
    conv_dropout_probs,
    pool_kernel_sizes,
    pool_strides,
    fc_sizes,
    fc_dropout_probs
)
lightning_model = Classifier(
    pytorch_model,
    num_classes=num_classes,
    learning_rate=learning_rate,
    weight_decay=weight_decay
)

trainer = L.Trainer()
tuner = Tuner(trainer)

lr_finder = tuner.lr_find(
    lightning_model,
    datamodule=dm,
    num_training=300,
    min_lr=1e-8,
    max_lr=1
)

print("Suggested LR:", lr_finder.suggestion())
fig = lr_finder.plot(suggest=True)
plt.show()